In [ ]:
# this notebook conduct the following task:
# 1 the origianl non redundant dataset was downloaded from here http://www.abybank.org/abdb/
#   \My Drive\@antibody\antigen_epitope\abdb_v2\data\abdb_data

# 2 Extract interaction info (tsv files) according to minumn atom-atom distance between sidechains of complexes
#   These tsv files have no SI information
#   structures_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/structures/"
#   output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info/"

# 3 Extract antigen's structure info and generate the mesh objs files using pymol
#   antigen_pdb_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdbs'
#   antigen_obj_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_objs'

# 4 add surface index to the antigen pdb file
#   SI_output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdb_with_SI"

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'

In [ ]:
import os
import shutil
from Bio import PDB

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'

# Initialize the PDB parser
parser = PDB.PDBParser()
io = PDB.PDBIO()

# Function to filter out non-standard amino acids and small molecules
def filter_residues(structure):
    for model in structure:
        for chain in list(model):
            for residue in list(chain):
                if not PDB.is_aa(residue, standard=True):
                    chain.detach_child(residue.id)
            if len(chain) == 0:  # remove empty chains
                model.detach_child(chain.id)

# Process each fold's training set
for fold in range(1, 6):
    train_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'abdb_data')
    processed_train_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'structures')

    os.makedirs(processed_train_dir, exist_ok=True)

    # Iterate over all PDB files in the training directory
    processed_count = 0
    for filename in os.listdir(train_dir):
        if filename.endswith(".pdb"):
            source_filepath = os.path.join(train_dir, filename)

            # Parse the PDB file
            structure = parser.get_structure('PDB_structure', source_filepath)

            # Filter out non-standard amino acids and small molecules
            filter_residues(structure)

            # Construct the new filename
            file_identifier = filename[:-4]  # remove '.pdb' extension
            chain_ids = [chain.id for model in structure for chain in model]
            new_filename = f"{file_identifier}_{'_'.join(chain_ids)}.pdb"
            target_filepath = os.path.join(processed_train_dir, new_filename)

            # Write the cleaned structure to the new file
            io.set_structure(structure)
            io.save(target_filepath)

            processed_count += 1

    print(f"Fold {fold}:")
    print(f"  Original training files: {len(os.listdir(train_dir))}")
    print(f"  Processed training files: {processed_count}")

print("Processing complete.")


In [ ]:

def count_files_in_folder(folder_path):
    items = os.listdir(folder_path)
    file_count = sum(1 for item in items if os.path.isfile(os.path.join(folder_path, item)))
    return file_count

In [ ]:
#@title Extract antigen's structure info and interaction info ==> minumn atom-atom distance between sidechains

from Bio import PDB
import pandas as pd
from pymol import cmd, stored
from scipy.spatial import distance
import os

# Function to calculate center of mass of sidechain
# Glycine (GLY) does not have a side chain beyond its alpha carbon, so its side chain center of mass would be undefined in your code
def calculate_center_of_mass(residue):
    if residue.resname == "GLY":
        return residue['CA'].coord  # Return alpha carbon coordinates for Glycine

    mass = 0
    x, y, z = 0, 0, 0
    backbone_atoms = ['N', 'CA', 'C', 'O']
    for atom in residue.get_unpacked_list():
        if atom.name not in backbone_atoms:
            mass += atom.mass
            x += atom.coord[0] * atom.mass
            y += atom.coord[1] * atom.mass
            z += atom.coord[2] * atom.mass
    if mass == 0:
        return 0, 0, 0
    return x / mass, y / mass, z / mass

# Function to check for heavy atom-atom interactions
def check_atom_atom_interaction(residue1, residue2, cutoff_distance=4.5, ca_cutoff_distance=15):
    # To save computational time, Check the CA-CA distance first
    if "CA" in residue1 and "CA" in residue2:
        ca_distance = distance.euclidean(residue1["CA"].coord, residue2["CA"].coord)
        if ca_distance > ca_cutoff_distance:
            return False  # If the CA-CA distance is greater than the threshold, no need to check atom-atom distances

    # Proceed with atom-atom distance calculations
    for atom1 in residue1.get_unpacked_list():
        if atom1.element == "H":  # skip hydrogen atoms
            continue
        for atom2 in residue2.get_unpacked_list():
            if atom2.element == "H":  # skip hydrogen atoms
                continue
            if distance.euclidean(atom1.coord, atom2.coord) <= cutoff_distance:
                return True  # found an interaction
    return False

# Process each fold's training set
for fold in range(1, 6):
    structures_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'structures')
    output_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_info')

    os.makedirs(output_dir, exist_ok=True)

    for pdb_file in os.listdir(structures_dir):
        if pdb_file.endswith('.pdb'):
            pdb_file_path = os.path.join(structures_dir, pdb_file)
            file_identifier = pdb_file[:-4]  # Remove the .pdb extension
            #print(file_identifier)  # For debugging

            # Extract chain information
            parts = file_identifier.split('_')
            pdb_id = parts[0]
            chains = parts[2:-1]  # Antibody chains are typically L and H
            antigen_chain = parts[-1]  # Antigen chains are the remaining parts

            # Load the PDB file into PyMOL
            cmd.reinitialize()  # Clear previous data
            cmd.load(pdb_file_path, "protein_complex")

            # Calculate relative SASA in the presence of antibody
            # SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
            # sasa_values = [value for key, value in SaSaRate.items()]

            # Initialize an empty list for secondary structure
            stored.ss_values = []
            cmd.iterate("protein_complex and chain " + antigen_chain + " and name CA", "stored.ss_values.append(ss)")

            # Hide or remove the antibody chains
            cmd.remove("chain " + "+".join(chains))  # Using the `remove` command to delete the antibody chains
            # Calculate relative SASA for the antigen chain in the absence of the antibody
            SaSaRate = cmd.get_sasa_relative("protein and chain " + antigen_chain)
            sasa_values = [value for key, value in SaSaRate.items()]

            # Initialize PDB Parser
            parser = PDB.PDBParser(QUIET=True)

            # Load structure
            structure = parser.get_structure(pdb_file.split('.')[0], pdb_file_path)

            # Initialize DataFrame
            columns = ['name', 'ResID', 'AA', 'SS', 'Cax', 'Cay', 'Caz', 'SCx', 'SCy', 'SCz', 'SaSaRate',
                      'G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
            df = pd.DataFrame(columns=columns)

            # Prepare a list for antibody residues
            antibody_residues = []
            for model in structure:
                for chain in model:
                    if chain.id in chains:
                        for residue in chain:
                            # Ensure that the residue is an amino acid
                            if residue.resname in PDB.Polypeptide.protein_letters_3to1:
                                antibody_residues.append(residue)

            # Iterate through structure to get antigen residues
            for model in structure:
                for chain in model:
                    if chain.id == antigen_chain:
                        for i, residue in enumerate(chain):
                            if "CA" in residue:
                                c_alpha = residue['CA'].coord
                                # Safeguard against IndexError
                                ss = stored.ss_values[i] if i < len(stored.ss_values) else 'Unknown'
                                sc_x, sc_y, sc_z = calculate_center_of_mass(residue)

                                interaction_data = {aa: 0 for aa in ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N', 'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']}
                                for ab_residue in antibody_residues:
                                    if check_atom_atom_interaction(residue, ab_residue):
                                        one_letter_code = PDB.Polypeptide.protein_letters_3to1[ab_residue.resname]
                                        interaction_data[one_letter_code] = 1

                                data = {'name': pdb_file.split('.')[0],
                                        'ResID': residue.id[1],  # Residue number
                                        'AA': residue.resname,
                                        'SS': ss,
                                        'Cax': c_alpha[0],
                                        'Cay': c_alpha[1],
                                        'Caz': c_alpha[2],
                                        'SCx': sc_x,
                                        'SCy': sc_y,
                                        'SCz': sc_z,
                                        'SaSaRate': sasa_values[i]}
                                data.update(interaction_data)
                                df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)

            # Save DataFrame to TSV
            output_path = os.path.join(output_dir, f"{pdb_file.split('.')[0]}_info.tsv")
            df.to_csv(output_path, sep='\t', index=False)

            print(f"Processed {pdb_file} and saved to {output_path}")

In [ ]:
for fold in range(1, 6):
    target_directory = base_fold_path + '/' + f'fold_{fold}' + '/data_test/antigen_info'

    print(count_files_in_folder(target_directory))

In [ ]:
#@title extract antigen chains and save

import os
import shutil
from Bio import PDB

# Initialize the PDB parser
parser = PDB.PDBParser()
io = PDB.PDBIO()

# Function to extract and save antigen chains
def save_antigen_chains(structure, source_filename, target_directory):
    for model in structure:
        # Create a new structure for antigens
        antigen_structure = PDB.Structure.Structure('antigen')

        # Copy model to antigen structure
        antigen_model = model.copy()
        antigen_structure.add(antigen_model)
        antigen_model.detach_parent()

        # Remove antibody chains (L and H) from antigen model
        for chain in list(antigen_model.get_chains()):
            if chain.id in ['L', 'H']:
                antigen_model.detach_child(chain.id)

        # If there are chains left (antigen), save the structure
        if len(list(antigen_model.get_chains())) > 0:
            new_filename = f"{source_filename[:-4]}_antigen.pdb"
            target_filepath = os.path.join(target_directory, new_filename)
            io.set_structure(antigen_structure)
            io.save(target_filepath)
            print(f"Saved antigen chains to {new_filename}")

# Process each fold's training set
for fold in range(1, 6):
    source_directory = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'structures')
    target_directory = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdbs')

    os.makedirs(target_directory, exist_ok=True)

    # Iterate over PDB files and process them
    for filename in os.listdir(source_directory):
        if filename.endswith(".pdb"):
            source_filepath = os.path.join(source_directory, filename)
            structure = parser.get_structure('PDB_structure', source_filepath)
            save_antigen_chains(structure, filename, target_directory)


In [ ]:
for fold in range(1, 6):
    target_directory = base_fold_path + '/' + f'fold_{fold}' + '/data_test/antigen_pdbs'

    print(count_files_in_folder(target_directory))

In [ ]:
#@title Generate antigen mesh obj files
# Create .obj file for each antigen pdb (run this code if antigen objs are not created)

from pymol import cmd

# Process each fold's training set
for fold in range(1, 6):
    antigen_pdb_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdbs')
    antigen_obj_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_objs')

    os.makedirs(antigen_obj_dir, exist_ok=True)

    if not os.path.exists(antigen_obj_dir):
      os.makedirs(antigen_obj_dir)
      print(f"Folder '{antigen_obj_dir}' was created.")
    else:
      print(f"Folder '{antigen_obj_dir}' already exists.")

    for pdb_filename in os.listdir(antigen_pdb_dir):
      if pdb_filename.endswith('.pdb'):
        # reset pymol
        cmd.reinitialize()

        pdb_file_path = os.path.join(antigen_pdb_dir, pdb_filename)

        # load pdb
        cmd.load(pdb_file_path, 'pdb_filename')
        cmd.hide('everything', 'pdb_filename')
        cmd.show('surface', 'pdb_filename')

        # set camera view
        cmd.set_view((
            1,    0,    0,
            0,    1,    0,
            0,    0,    1,
            0,    0,    0,
            0,    0,    0,
            0,    300,  1
        ))

        # save obj file
        obj_filename = pdb_filename.replace('.pdb', '.obj')
        cmd.save(os.path.join(antigen_obj_dir, obj_filename), 'pdb_filename', format='obj')


In [ ]:
for fold in range(1, 6):
    target_directory = base_fold_path + '/' + f'fold_{fold}' + '/data_test/antigen_objs'

    print(count_files_in_folder(target_directory))

In [ ]:
#@title Generate antigen pdb with SI

In [ ]:
!pip install pywavefront
!pip install ipyvolume
!pip install pyvista
!pip install matplotlib ipywidgets

In [ ]:
!pip install trimesh
!pip install rtree

In [ ]:
import ipyvolume as ipv
from matplotlib import cm
import pyvista as pv
import pywavefront
import os

In [ ]:
#@title Add Shape Index (yuhao method 4)
import numpy as np
import trimesh
import os
import pandas as pd
from Bio import PDB
from scipy.spatial import KDTree

# Process each fold's training set
for fold in range(1, 6):
    antigen_pdb_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdbs')
    antigen_obj_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_objs')
    SI_output_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdb_with_SI')

    os.makedirs(SI_output_dir, exist_ok=True)

    for obj_filename in os.listdir(antigen_obj_dir):
        if obj_filename.endswith('.obj'):
            mesh = trimesh.load_mesh(os.path.join(antigen_obj_dir, obj_filename))
            vertices = mesh.vertices
            faces = mesh.faces

            K = trimesh.curvature.discrete_gaussian_curvature_measure(mesh, mesh.vertices, 2)
            H = trimesh.curvature.discrete_mean_curvature_measure(mesh, mesh.vertices, 2)
            elem = np.square(H) - K
            elem[elem <= 0] = 1e-8
            k1 = H + np.sqrt(elem)
            k2 = H - np.sqrt(elem)
            shape_index = (2.0 / np.pi) * np.arctan((k1 + k2) / (k1 - k2))

            obj_coords = mesh.vertices
            obj_curvature = shape_index

            pdb_filename = os.path.splitext(obj_filename)[0] + '.pdb'
            pdb_filepath = os.path.join(antigen_pdb_dir, pdb_filename)

            parser = PDB.PDBParser()
            structure = parser.get_structure("protein", pdb_filepath)

            pdb_coords = []
            for atom in structure.get_atoms():
                pdb_coords.append(atom.get_coord())
            pdb_coords = np.array(pdb_coords)

            kd_tree = KDTree(obj_coords)
            distances, indices = kd_tree.query(pdb_coords)

            atom_index = 0
            for atom in structure.get_atoms():
                atom.set_bfactor(shape_index[indices[atom_index]])
                atom_index += 1

            output_pdb_filename = os.path.splitext(pdb_filename)[0] + '_updated.pdb'
            output_pdb_filepath = os.path.join(SI_output_dir, output_pdb_filename)

            io = PDB.PDBIO()
            io.set_structure(structure)
            io.save(output_pdb_filepath)

In [ ]:


from Bio import PDB

import pandas as pd

# Initialize PDB parser
pdb_parser = PDB.PDBParser()

# Path to your PDB file with Shape Index stored in the B-factor column
pdb_file_path = '/content/drive/MyDrive/abdb_felipe/step1_part1/fold_1/data_test/antigen_pdb_with_SI/4H88_1_L_H_A_antigen_updated.pdb'

# Parse the PDB file
structure = pdb_parser.get_structure('antigen_with_SI', pdb_file_path)

# Extract atom details including the B-factor (which now contains the SI)
atoms_data = []
for model in structure:
    for chain in model:
        for residue in chain:
            for atom in residue:
                atoms_data.append({
                    'atom_name': atom.get_name(),
                    'residue': residue.get_resname(),
                    'residue_id': residue.get_id(),
                    'chain_id': chain.id,
                    'x_coord': atom.coord[0],
                    'y_coord': atom.coord[1],
                    'z_coord': atom.coord[2],
                    'occupancy': atom.get_occupancy(),
                    'b_factor': atom.get_bfactor()  # SI is stored in the B-factor field
                })

# Convert the data to a pandas DataFrame for easy visualization
atoms_df = pd.DataFrame(atoms_data)

# Display the dataframe
atoms_df.head()  # Show the first few rows of the DataFrame


In [ ]:
import numpy as np
import trimesh
import os
import pandas as pd
from Bio import PDB
from scipy.spatial import KDTree

# Process each fold's training set
for fold in range(1, 6):
    antigen_pdb_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdbs')
    antigen_obj_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_objs')
    SI_output_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_test', 'antigen_pdb_with_SI')

    os.makedirs(SI_output_dir, exist_ok=True)

    # List all the processed PDB files
    processed_files = [f for f in os.listdir(SI_output_dir) if f.endswith('_updated.pdb')]

    # Extract the base names of the processed files
    processed_basenames = [os.path.splitext(f)[0].replace('_updated', '') for f in processed_files]

    # Process only the unprocessed .obj files
    for obj_filename in os.listdir(antigen_obj_dir):
        if obj_filename.endswith('.obj'):
            base_name = os.path.splitext(obj_filename)[0]

            # Skip if the corresponding PDB file has already been processed
            if base_name in processed_basenames:
                continue

            try:
                mesh = trimesh.load_mesh(os.path.join(antigen_obj_dir, obj_filename))
                vertices = mesh.vertices
                faces = mesh.faces

                K = trimesh.curvature.discrete_gaussian_curvature_measure(mesh, mesh.vertices, 2)
                H = trimesh.curvature.discrete_mean_curvature_measure(mesh, mesh.vertices, 2)
                elem = np.square(H) - K
                elem[elem <= 0] = 1e-8
                k1 = H + np.sqrt(elem)
                k2 = H - np.sqrt(elem)
                shape_index = (2.0 / np.pi) * np.arctan((k1 + k2) / (k1 - k2))

                obj_coords = mesh.vertices
                obj_curvature = shape_index

                pdb_filename = os.path.splitext(obj_filename)[0] + '.pdb'
                pdb_filepath = os.path.join(antigen_pdb_dir, pdb_filename)

                parser = PDB.PDBParser()
                structure = parser.get_structure("protein", pdb_filepath)

                pdb_coords = []
                for atom in structure.get_atoms():
                    pdb_coords.append(atom.get_coord())
                pdb_coords = np.array(pdb_coords)

                kd_tree = KDTree(obj_coords)
                distances, indices = kd_tree.query(pdb_coords)

                atom_index = 0
                for atom in structure.get_atoms():
                    atom.set_bfactor(shape_index[indices[atom_index]])
                    atom_index += 1

                output_pdb_filename = os.path.splitext(pdb_filename)[0] + '_updated.pdb'
                output_pdb_filepath = os.path.join(SI_output_dir, output_pdb_filename)

                io = PDB.PDBIO()
                io.set_structure(structure)
                io.save(output_pdb_filepath)

            except Exception as e:
                print(f"Error processing {obj_filename}: {e}")

In [ ]:

for fold in range(1, 6):
    target_directory = base_fold_path + '/' + f'fold_{fold}' + '/data_test/antigen_pdb_with_SI'

    print(count_files_in_folder(target_directory))